# FixIt Madison: YOLO26-OBB ceiling-damage proof of concept (Colab GPU)
Small isolated experiment: 150 train / 50 valid / 50 test images from Roboflow "Ceiling" v1 (CC BY 4.0,
universe.roboflow.com/scipublic/ceiling-y9dkm). Ultralytics 8.4.196 + pretrained `yolo26n-obb.pt` (AGPL-3.0).
Nothing here touches the FixIt app or Databricks.

1. Runtime > Change runtime type > **T4 GPU** (free tier).
2. Upload `yolo_ceiling_bundle.zip` (made by `make_colab_bundle.py`) when asked.
3. Run all. At the end, download `yolo_ceiling_results.zip` and unzip it into `experiments/yolo_ceiling/`.
The test split is evaluated **once**, after training; it is never used for tuning or checkpoint selection.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q ultralytics==8.4.196 mlflow

In [ ]:
from google.colab import files
up = files.upload()            # choose yolo_ceiling_bundle.zip
import zipfile; zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd /content/yolo_ceiling
!ls dataset_250 && for s in train valid test; do echo $s $(ls dataset_250/$s/images | wc -l); done

In [ ]:
!python train_yolo.py --model yolo26n-obb.pt     # max 50 epochs, patience 10, seed 42, imgsz 640

Optional comparison (only if time allows; ~2-3x slower):

In [ ]:
# !python train_yolo.py --model yolo26s-obb.pt

In [ ]:
!python evaluate.py --weights runs/yolo26n-obb/weights/best.pt
# !python evaluate.py --weights runs/yolo26s-obb/weights/best.pt

In [ ]:
import shutil, glob
shutil.make_archive('/content/yolo_ceiling_results', 'zip', '/content/yolo_ceiling',
                    base_dir=None)  # includes runs/ (best.pt, curves) and results/
files.download('/content/yolo_ceiling_results.zip')